# Imports and project paths

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Raw data directory exists:", RAW_DIR.exists())

Project root: c:\Users\Acer\Desktop\customer-churn-retention-mvp
Raw data directory: c:\Users\Acer\Desktop\customer-churn-retention-mvp\data\raw
Raw data directory exists: True


# Load selected datasets

In [3]:
dataset_paths = {
    "telco": RAW_DIR / "Telco-Customer-Churn.csv",
    "olist_customers": RAW_DIR / "olist_customers_dataset.csv",
    "olist_orders": RAW_DIR / "olist_orders_dataset.csv",
    "olist_order_items": RAW_DIR / "olist_order_items_dataset.csv",
    "olist_payments": RAW_DIR / "olist_order_payments_dataset.csv",
    "olist_reviews": RAW_DIR / "olist_order_reviews_dataset.csv",
    "olist_products": RAW_DIR / "olist_products_dataset.csv",
    "olist_sellers": RAW_DIR / "olist_sellers_dataset.csv",
    "olist_geolocation": RAW_DIR / "olist_geolocation_dataset.csv",
    "product_category_translation": RAW_DIR / "product_category_name_translation.csv",
}

datasets = {}
missing_files = []

for name, path in dataset_paths.items():
    if path.exists():
        datasets[name] = pd.read_csv(path)
        print(f"{name}: loaded {datasets[name].shape[0]:,} rows × {datasets[name].shape[1]} columns")
    else:
        missing_files.append(str(path))
        print(f"FILE NOT FOUND: {path}")

if missing_files:
    print("\nCheck the file names and ensure all required CSV files are inside data/raw.")

telco: loaded 7,043 rows × 21 columns
olist_customers: loaded 99,441 rows × 5 columns
olist_orders: loaded 99,441 rows × 8 columns
olist_order_items: loaded 112,650 rows × 7 columns
olist_payments: loaded 103,886 rows × 5 columns
olist_reviews: loaded 99,224 rows × 7 columns
olist_products: loaded 32,951 rows × 9 columns
olist_sellers: loaded 3,095 rows × 4 columns
olist_geolocation: loaded 1,000,163 rows × 5 columns
product_category_translation: loaded 71 rows × 2 columns


# Dataset overview

In [4]:
overview = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2)
    }
    for name, df in datasets.items()
])
display(overview)

for name, df in datasets.items():
    print(f"\n{name} — column names and data types")
    display(df.dtypes.rename("data_type").to_frame())

,dataset,rows,columns,memory_mb
0,telco,7043,21,6.82
1,olist_customers,99441,5,26.59
2,olist_orders,99441,8,52.94
3,olist_order_items,112650,7,35.99
4,olist_payments,103886,5,16.23
5,olist_reviews,99224,7,39.12
6,olist_products,32951,9,6.30
7,olist_sellers,3095,4,0.59
8,olist_geolocation,1000163,5,129.38
9,product_category_translation,71,2,0.01



telco — column names and data types


,data_type
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object



olist_customers — column names and data types


,data_type
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object



olist_orders — column names and data types


... (11 outputs left out)


olist_geolocation — column names and data types


,data_type
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,object
geolocation_state,object



product_category_translation — column names and data types


,data_type
product_category_name,object
product_category_name_english,object


# Missing-value analysis

In [5]:
missing_summary = []

for name, df in datasets.items():
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = int(df.isna().sum().sum())
    missing_summary.append({
        "dataset": name,
        "missing_cells": missing_cells,
        "missing_percentage": round(missing_cells / total_cells * 100, 2) if total_cells else 0
    })

display(pd.DataFrame(missing_summary))

for name, df in datasets.items():
    column_missing = df.isna().sum()
    column_missing = column_missing[column_missing > 0].sort_values(ascending=False)
    if not column_missing.empty:
        print(f"\n{name}: columns containing missing values")
        display(column_missing.rename("missing_count").to_frame())

,dataset,missing_cells,missing_percentage
0,telco,0,0.00
1,olist_customers,0,0.00
2,olist_orders,4908,0.62
3,olist_order_items,0,0.00
4,olist_payments,0,0.00
5,olist_reviews,145903,21.01
6,olist_products,2448,0.83
7,olist_sellers,0,0.00
8,olist_geolocation,0,0.00
9,product_category_translation,0,0.00



olist_orders: columns containing missing values


,missing_count
order_delivered_customer_date,2965
order_delivered_carrier_date,1783
order_approved_at,160



olist_reviews: columns containing missing values


,missing_count
review_comment_title,87656
review_comment_message,58247



olist_products: columns containing missing values


,missing_count
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


# Duplicate-row analysis

In [6]:
duplicate_summary = pd.DataFrame([
    {"dataset": name, "exact_duplicate_rows": int(df.duplicated().sum())}
    for name, df in datasets.items()
])
display(duplicate_summary)

# Duplicates are reported only; no rows are deleted in this notebook.

,dataset,exact_duplicate_rows
0,telco,0
1,olist_customers,0
2,olist_orders,0
3,olist_order_items,0
4,olist_payments,0
5,olist_reviews,0
6,olist_products,0
7,olist_sellers,0
8,olist_geolocation,261831
9,product_category_translation,0


# Telco churn class balance

In [7]:
telco = datasets["telco"]

churn_summary = (
    telco["Churn"].value_counts(dropna=False)
    .rename_axis("churn_status")
    .reset_index(name="customer_count")
)
churn_summary["percentage"] = (churn_summary["customer_count"] / len(telco) * 100).round(2)
display(churn_summary)

,churn_status,customer_count,percentage
0,No,5174,73.46
1,Yes,1869,26.54


# Validate key Olist table relationships

In [8]:
customers = datasets["olist_customers"]
orders = datasets["olist_orders"]
order_items = datasets["olist_order_items"]
payments = datasets["olist_payments"]
reviews = datasets["olist_reviews"]
products = datasets["olist_products"]
sellers = datasets["olist_sellers"]

relationship_summary = pd.DataFrame([
    {
        "relationship": "orders.customer_id -> customers.customer_id",
        "unmatched_rows": int((~orders["customer_id"].isin(customers["customer_id"])).sum())
    },
    {
        "relationship": "order_items.order_id -> orders.order_id",
        "unmatched_rows": int((~order_items["order_id"].isin(orders["order_id"])).sum())
    },
    {
        "relationship": "order_items.product_id -> products.product_id",
        "unmatched_rows": int((~order_items["product_id"].isin(products["product_id"])).sum())
    },
    {
        "relationship": "order_items.seller_id -> sellers.seller_id",
        "unmatched_rows": int((~order_items["seller_id"].isin(sellers["seller_id"])).sum())
    },
    {
        "relationship": "payments.order_id -> orders.order_id",
        "unmatched_rows": int((~payments["order_id"].isin(orders["order_id"])).sum())
    },
    {
        "relationship": "reviews.order_id -> orders.order_id",
        "unmatched_rows": int((~reviews["order_id"].isin(orders["order_id"])).sum())
    }
])
display(relationship_summary)

key_checks = pd.DataFrame([
    {"dataset": "olist_customers", "key": "customer_id",
     "null_keys": int(customers["customer_id"].isna().sum()),
     "duplicate_key_rows": int(customers["customer_id"].duplicated().sum())},
    {"dataset": "olist_orders", "key": "order_id",
     "null_keys": int(orders["order_id"].isna().sum()),
     "duplicate_key_rows": int(orders["order_id"].duplicated().sum())},
    {"dataset": "olist_order_items", "key": "order_id + order_item_id",
     "null_keys": int(order_items[["order_id", "order_item_id"]].isna().any(axis=1).sum()),
     "duplicate_key_rows": int(order_items.duplicated(["order_id", "order_item_id"]).sum())},
    {"dataset": "olist_products", "key": "product_id",
     "null_keys": int(products["product_id"].isna().sum()),
     "duplicate_key_rows": int(products["product_id"].duplicated().sum())},
    {"dataset": "olist_sellers", "key": "seller_id",
     "null_keys": int(sellers["seller_id"].isna().sum()),
     "duplicate_key_rows": int(sellers["seller_id"].duplicated().sum())},
])
display(key_checks)


,relationship,unmatched_rows
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0
3,order_items.seller_id -> sellers.seller_id,0
4,payments.order_id -> orders.order_id,0
5,reviews.order_id -> orders.order_id,0


,dataset,key,null_keys,duplicate_key_rows
0,olist_customers,customer_id,0,0
1,olist_orders,order_id,0,0
2,olist_order_items,order_id + order_item_id,0,0
3,olist_products,product_id,0,0
4,olist_sellers,seller_id,0,0
